In [ ]:
!pip install tqdm
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from tqdm import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Process ALL 75K products
print("="*70)
print("EXTRACTING CLIP EMBEDDINGS - FULL DATASET")
print("="*70)

embeddings_list = []
prices_list = []
ids_list = []
failed = 0

batch_size = 16  # CLIP needs smaller batches
total = len(df)

batch_images = []
batch_texts = []
batch_data = []

pbar = tqdm(total=total, desc="CLIP extraction")

for idx, row in df.iterrows():
    img_path = Path(IMAGE_DIR) / row['image_filename']

    if not img_path.exists():
        failed += 1
        pbar.update(1)
        continue

    # Prepare data
    try:
        img = Image.open(img_path).convert('RGB')
        text = str(row.get('title', '')) + ' ' + str(row.get('description', ''))
        text = text[:500]  # Truncate

        batch_images.append(img)
        batch_texts.append(text)
        batch_data.append({'price': row['price'], 'id': row['id']})
    except:
        failed += 1
        pbar.update(1)
        continue

    # Process when batch is full
    if len(batch_images) == batch_size:
        try:
            inputs = processor(
                text=batch_texts,
                images=batch_images,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=77
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                img_embeds = outputs.image_embeds.cpu().numpy()
                txt_embeds = outputs.text_embeds.cpu().numpy()

            # Concatenate image + text
            combined = np.concatenate([img_embeds, txt_embeds], axis=1)
            embeddings_list.append(combined)

            for data in batch_data:
                prices_list.append(data['price'])
                ids_list.append(data['id'])

        except Exception as e:
            print(f"\nBatch error: {e}")
            failed += len(batch_images)

        # Clear batch
        batch_images = []
        batch_texts = []
        batch_data = []
        pbar.update(batch_size)

        # Free memory
        torch.cuda.empty_cache()

# Process remaining
if batch_images:
    try:
        inputs = processor(text=batch_texts, images=batch_images, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            img_embeds = outputs.image_embeds.cpu().numpy()
            txt_embeds = outputs.text_embeds.cpu().numpy()
        combined = np.concatenate([img_embeds, txt_embeds], axis=1)
        embeddings_list.append(combined)
        for data in batch_data:
            prices_list.append(data['price'])
            ids_list.append(data['id'])
    except:
        failed += len(batch_images)
    pbar.update(len(batch_images))

pbar.close()

# Combine
embeddings = np.vstack(embeddings_list)
prices = np.array(prices_list)
ids = np.array(ids_list)

print(f"\n✓ Done!")
print(f"  Processed: {len(embeddings)}/{total}")
print(f"  Failed: {failed}")
print(f"  Shape: {embeddings.shape}")

In [ ]:
# ============================================
# CELL 6: Log-Transform Prices (CRITICAL!)
# ============================================

print("="*70)
print("LOG-TRANSFORMING PRICES")
print("="*70)

# Original prices
print(f"Original price range: ${prices.min():.2f} - ${prices.max():.2f}")
print(f"Mean: ${prices.mean():.2f}")
print(f"Std: ${prices.std():.2f}")

# Log-transform (this prevents explosions!)
prices_log = np.log(prices + 1)  # +1 to avoid log(0)

print(f"\nLog-transformed range: {prices_log.min():.4f} - {prices_log.max():.4f}")
print(f"Mean: {prices_log.mean():.4f}")
print(f"Std: {prices_log.std():.4f}")

# Save BOTH versions
data = {
    'embeddings': embeddings,
    'prices': prices,              # Original prices
    'prices_log': prices_log,      # Log prices (use this for training!)
    'product_ids': ids,
    'embedding_dim': 1024,
    'model': 'CLIP-ViT-B/32'
}
output_file = 'clip_embeddings_log.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(data, f)

print(f"\n✓ Saved with log-transformed prices!")
import shutil
drive_path = '/content/drive/MyDrive/amazon/'
shutil.copy(output_file, drive_path)
print(f"✓ Backed up to Google Drive")

In [ ]:
file_path = '/content/drive/MyDrive/amazon/clip_embeddings_log.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

embeddings = np.array(data['embeddings'])
prices_log = np.array(data['prices_log'])  # use for training
prices = np.array(data['prices'])          # use for inverse transform at the end
ids = np.array(data['product_ids'])


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    embeddings, prices_log, test_size=0.1, random_state=42
)


In [ ]:
import pickle
import numpy as np

# Load your embeddings file
file_path = '/content/drive/MyDrive/amazon/clip_embeddings_log.pkl'
with open(file_path, 'rb') as f:
    data = pickle.load(f)

# Assign variables
X = np.array(data['embeddings'])      # CLIP embeddings (text + image combined)
y_log = np.array(data['prices_log'])  # Log-transformed prices
y = np.array(data['prices'])          # Original prices, for inverse log later
ids = np.array(data['product_ids'])

print("X shape:", X.shape)
print("y_log shape:", y_log.shape)


X shape: (75000, 1024)
y_log shape: (75000,)


In [ ]:
!pip uninstall lightgbm

Found existing installation: lightgbm 4.6.0
Uninstalling lightgbm-4.6.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/lightgbm-4.6.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/lightgbm/*
Proceed (Y/n)? Y
  Successfully uninstalled lightgbm-4.6.0


In [ ]:
!pip install lightgbm --config-settings=cmake.args="-DUSE_GPU=1"

In [ ]:
# Check GPU availability
import lightgbm as lgb
import numpy as np

print("LightGBM version:", lgb.__version__)

# Create small test dataset
X_test = np.random.rand(1000, 10)
y_test = np.random.rand(1000)

try:
    test_data = lgb.Dataset(X_test, label=y_test)
    test_params = {
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
        'verbose': 1
    }
    model = lgb.train(test_params, test_data, num_boost_round=10)
    print("✓ GPU training successful!")
except Exception as e:
    print(f"✗ GPU training failed: {e}")

LightGBM version: 4.6.0
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 10
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 10 dense feature groups (0.01 MB) transferred to GPU in 0.000564 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score 0.500915
✓ GPU training successful!


In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

X_train, X_val, y_train, y_val = train_test_split(X, y_log, test_size=0.1, random_state=42)

# More explicit GPU parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 1,  # Changed to see more output
    'device': 'gpu',
    'gpu_use_dp': False,  # Use single precision for faster training
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_bin': 63  # GPU works better with power of 2 minus 1
}

print("Creating datasets...")
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

print("Training LightGBM with GPU acceleration...")
print(f"Training data shape: {X_train.shape}")
print(f"Device: {params['device']}")

model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    num_boost_round=10000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(100)
    ]
)

# Save model to Drive
model_save_path = "/content/drive/MyDrive/amazon/lgb_model.txt"
lgb_model.save_model(model_save_path)
print(f"✅ Model retrained and saved at {model_save_path}")


print(f"\nBest iteration: {model.best_iteration}")
print(f"Best score: {model.best_score}")

Creating datasets...
Training LightGBM with GPU acceleration...
Training data shape: (67500, 1024)
Device: gpu
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 64512
[LightGBM] [Info] Number of data points in the train set: 67500, number of used features: 1024
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 64 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1024 dense feature groups (65.92 MB) transferred to GPU in 0.084359 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score 2.738202
Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.704478	valid's rmse: 0.779282
[200]	train's rmse: 0.628152	valid's rmse: 0.756877
[300]	train's rmse: 0.56945	valid's rmse: 0.745643
[400]	train's rmse: 0.520746	valid's rmse

In [ ]:
# ======================================================
# CLIP Image Embeddings for Test Dataset
# ======================================================

!pip install ftfy regex tqdm git+https://github.com/openai/CLIP.git > /dev/null

import torch
import clip
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
from io import BytesIO
import pickle

# -----------------------------
# 1. Load CLIP model
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# -----------------------------
# 2. Load your test CSV
# -----------------------------
csv_path = "/content/sample_test.csv"  # <-- change if needed
df = pd.read_csv(csv_path)

# Identify your image column name
print("Columns in CSV:", df.columns.tolist())
# Let's assume it's called 'image_link' (change if different)

# -----------------------------
# 3. Define helper to load image from URL
# -----------------------------
def load_image_from_url(url):
    try:
        response = requests.get(url, timeout=10)
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image
    except Exception as e:
        print(f"Error loading image {url}: {e}")
        return None

# -----------------------------
# 4. Generate image embeddings
# -----------------------------
all_embeddings = []
valid_ids = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    img = load_image_from_url(row["image_link"])
    if img is None:
        continue

    image_input = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        img_features = model.encode_image(image_input)
        img_features /= img_features.norm(dim=-1, keepdim=True)

    all_embeddings.append(img_features.cpu().numpy())
    valid_ids.append(row.get("sample_id", i))

# -----------------------------
# 5. Save embeddings
# -----------------------------
embeddings_array = np.vstack(all_embeddings)
data = {"embeddings": embeddings_array, "sample_id": valid_ids}

save_path = "/content/clip_image_embeddings_test.pkl"
with open(save_path, "wb") as f:
    pickle.dump(data, f)

print(f"\n✅ Saved {len(embeddings_array)} embeddings to {save_path}")


  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-alp1mn8w


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 102MiB/s]


Columns in CSV: ['sample_id', 'catalog_content', 'image_link']


100%|██████████| 100/100 [00:10<00:00,  9.54it/s]


✅ Saved 100 embeddings to /content/clip_image_embeddings_test.pkl


In [ ]:
with open('/content/clip_image_embeddings_test.pkl', 'rb') as f:
    test_data = pickle.load(f)

test_embeddings = np.array(test_data['embeddings'])
test_ids = np.array(test_data['sample_id'])

test_preds_log = model.predict(test_embeddings)
test_preds = np.expm1(test_preds_log)


KeyError: 'product_ids'

In [ ]:
model_save_path = "/content/drive/MyDrive/amazon/lgb_model.txt"
model.save_model(model_save_path)
print(f"✅ Model saved at {model_save_path}")


AttributeError: 'CLIP' object has no attribute 'save_model'

In [ ]:
whos


Variable              Type              Data/Info
-------------------------------------------------
BytesIO               type              <class '_io.BytesIO'>
Image                 module            <module 'PIL.Image' from <...>t-packages/PIL/Image.py'>
X                     ndarray           75000x1024: 76800000 elems, type `float32`, 307200000 bytes (292.96875 Mb)
X_test                ndarray           1000x10: 10000 elems, type `float64`, 80000 bytes
X_train               ndarray           67500x1024: 69120000 elems, type `float32`, 276480000 bytes (263.671875 Mb)
X_val                 ndarray           7500x1024: 7680000 elems, type `float32`, 30720000 bytes (29.296875 Mb)
all_embeddings        list              n=100
clip                  module            <module 'clip' from '/usr<...>ckages/clip/__init__.py'>
csv_path              str               /content/sample_test.csv
data                  dict              n=2
device                str               cuda
df           

In [ ]:
# ======================================================
# RETRAIN LIGHTGBM MODEL AND SAVE TO DRIVE
# ======================================================

import lightgbm as lgb

lgb_model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(50)],
)

# Save model to Drive
model_save_path = "/content/drive/MyDrive/amazon/lgb_model.txt"
lgb_model.save_model(model_save_path)
print(f"✅ Model retrained and saved at {model_save_path}")


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 64512
[LightGBM] [Info] Number of data points in the train set: 67500, number of used features: 1024
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 64 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 1024 dense feature groups (65.92 MB) transferred to GPU in 0.077780 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score 2.738202
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[999]	training's rmse: 0.333771	valid_1's rmse: 0.7234
✅ Model retrained and saved at /content/drive/MyDrive/amazon/lgb_model.txt


In [ ]:
# ======================================================
# CLIP Image & Text Embeddings for Test Dataset
# ======================================================

!pip install ftfy regex tqdm git+https://github.com/openai/CLIP.git > /dev/null

import torch
import clip
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
from io import BytesIO
import pickle

# -----------------------------
# 1. Load CLIP model
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# -----------------------------
# 2. Load your test CSV
# -----------------------------
csv_path = "/content/sample_test.csv"  # <-- Change if needed
df = pd.read_csv(csv_path)

# --- Configuration ---
# Identify your image and text column names
image_column = "image_link"      # <-- Change if your image URL column is different
text_column = "catalog_content"  # <-- Change if your text column is different
id_column = "sample_id"          # <-- Change if your ID column is different

print("Columns in CSV:", df.columns.tolist())
print(f"Using '{image_column}' for images and '{text_column}' for text.")

# -----------------------------
# 3. Define helper to load image from URL
# -----------------------------
def load_image_from_url(url):
    """Fetches and opens an image from a URL."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an exception for bad status codes
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image
    except Exception as e:
        print(f"Error loading image {url}: {e}")
        return None

# -----------------------------
# 4. Generate image and text embeddings
# -----------------------------
all_image_embeddings = []
all_text_embeddings = []
valid_ids = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    # --- Process Image ---
    img = load_image_from_url(row[image_column])
    if img is None:
        continue # Skip row if image can't be loaded

    image_input = preprocess(img).unsqueeze(0).to(device)

    # --- Process Text ---
    text_description = str(row[text_column]) # Ensure text is a string
    text_input = clip.tokenize([text_description], truncate=True).to(device)

    with torch.no_grad():
        # Encode both image and text
        img_features = model.encode_image(image_input)
        text_features = model.encode_text(text_input)

        # Normalize the features
        img_features /= img_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

    # Append successful embeddings and ID
    all_image_embeddings.append(img_features.cpu().numpy())
    all_text_embeddings.append(text_features.cpu().numpy())
    valid_ids.append(row.get(id_column, i))

# -----------------------------
# 5. Save embeddings
# -----------------------------
# Convert lists of arrays into 2D numpy arrays
image_embeddings_array = np.vstack(all_image_embeddings)
text_embeddings_array = np.vstack(all_text_embeddings)

# Store both embeddings in a dictionary
data = {
    "image_embeddings": image_embeddings_array,
    "text_embeddings": text_embeddings_array,
    "sample_id": valid_ids
}

save_path = "/content/clip_multimodal_embeddings_test.pkl"
with open(save_path, "wb") as f:
    pickle.dump(data, f)

print(f"\n✅ Saved {len(valid_ids)} image and text embedding pairs to {save_path}")

  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-nspwvwe9
Columns in CSV: ['sample_id', 'catalog_content', 'image_link']
Using 'image_link' for images and 'catalog_content' for text.


100%|██████████| 100/100 [00:10<00:00,  9.71it/s]


✅ Saved 100 image and text embedding pairs to /content/clip_multimodal_embeddings_test.pkl


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
from google.colab import drive

# -----------------------------
# 1. Mount Google Drive
# -----------------------------
# This will prompt you for authorization.
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("✅ Drive mounted successfully!")


# -----------------------------
# 2. Define File Paths
# -----------------------------
# --- IMPORTANT: Please verify these paths are correct! ---

# Path to the embeddings file you created in the previous step
embeddings_path = "/content/clip_multimodal_embeddings_test.pkl"

# Path to your saved model in Google Drive
# Make sure "amazon folder" exactly matches the folder name in your Drive.
model_path = "/content/drive/MyDrive/amazon/lgb_model.txt"

# Path for the output predictions file
output_path = "/content/price_predictions.csv"


# -----------------------------
# 3. Load Embeddings
# -----------------------------
print(f"\nLoading embeddings from {embeddings_path}...")
with open(embeddings_path, 'rb') as f:
    data = pickle.load(f)

image_embeddings = data["image_embeddings"]
text_embeddings = data["text_embeddings"]
sample_ids = data["sample_id"]

print(f"Loaded {len(sample_ids)} embedding pairs.")
print("Image embedding shape:", image_embeddings.shape)
print("Text embedding shape:", text_embeddings.shape)


# -----------------------------
# 4. Combine Embeddings into a Single Feature Set
# -----------------------------
# Most models require a single input matrix. We concatenate the image
# and text features side-by-side for each sample.
print("\nCombining image and text embeddings into a single feature matrix...")
X_test = np.concatenate([image_embeddings, text_embeddings], axis=1)

print("Combined feature matrix shape:", X_test.shape)


# -----------------------------
# 5. Load the Pre-trained LightGBM Model
# -----------------------------
print(f"\nLoading LightGBM model from {model_path}...")
try:
    bst = lgb.Booster(model_file=model_path)
    print("✅ Model loaded successfully!")
except lgb.basic.LightGBMError as e:
    print(f"\n❌ ERROR: Could not load the model. Please check the following:")
    print(f"1. Does the file exist at '{model_path}'?")
    print(f"2. Is the folder name 'amazon folder' spelled correctly (it's case-sensitive)?")
    print(f"3. Was the model saved correctly as a LightGBM '.txt' file?")
    raise e


# -----------------------------
# 6. Make Predictions
# -----------------------------
print("\nGenerating price predictions...")
predicted_prices = bst.predict(X_test)
print("✅ Predictions generated.")


# -----------------------------
# 7. Save Predictions to CSV
# -----------------------------
print(f"\nSaving predictions to {output_path}...")
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predicted_prices
})

results_df.to_csv(output_path, index=False)

print(f"\n🎉 Success! Predictions saved. You can find the file at '{output_path}'")
print("\nFirst 5 predictions:")
print(results_df.head())

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted successfully!

Loading embeddings from /content/clip_multimodal_embeddings_test.pkl...
Loaded 100 embedding pairs.
Image embedding shape: (100, 512)
Text embedding shape: (100, 512)

Combining image and text embeddings into a single feature matrix...
Combined feature matrix shape: (100, 1024)

Loading LightGBM model from /content/drive/MyDrive/amazon/lgb_model.txt...
✅ Model loaded successfully!

Generating price predictions...
✅ Predictions generated.

Saving predictions to /content/price_predictions.csv...

🎉 Success! Predictions saved. You can find the file at '/content/price_predictions.csv'

First 5 predictions:
   sample_id  predicted_price
0     217392         3.940823
1     209156         3.056504
2     262333         1.820536
3     295979         2.292511
4      50604         2.398281


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
from google.colab import drive

# -----------------------------
# 1. Mount Google Drive
# -----------------------------
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True) # Added force_remount
print("✅ Drive mounted successfully!")


# -----------------------------
# 2. Define File Paths
# -----------------------------
embeddings_path = "/content/clip_multimodal_embeddings_test.pkl"
model_path = "/content/drive/MyDrive/amazon/lgb_model.txt"
output_path = "/content/price_predictions.csv"


# -----------------------------
# 3. Load Embeddings
# -----------------------------
print(f"\nLoading embeddings from {embeddings_path}...")
with open(embeddings_path, 'rb') as f:
    data = pickle.load(f)

image_embeddings = data["image_embeddings"]
text_embeddings = data["text_embeddings"]
sample_ids = data["sample_id"]
print(f"Loaded {len(sample_ids)} embedding pairs.")


# -----------------------------
# 4. Combine Embeddings into a Single Feature Set
# -----------------------------
print("\nCombining image and text embeddings...")
X_test = np.concatenate([image_embeddings, text_embeddings], axis=1)
print("Combined feature matrix shape:", X_test.shape)


# -----------------------------
# 5. Load the Pre-trained LightGBM Model
# -----------------------------
print(f"\nLoading LightGBM model from {model_path}...")
bst = lgb.Booster(model_file=model_path)
print("✅ Model loaded successfully!")


# -----------------------------
# 6. Make Predictions & Convert from Log Scale
# -----------------------------
print("\nGenerating log predictions...")
log_predictions = bst.predict(X_test)

# --- THIS IS THE NEW LINE ---
# Convert log predictions back to the original price scale
# We use np.expm1 which is the inverse of np.log1p
print("Converting predictions from log scale to actual prices...")
predicted_prices = np.expm1(log_predictions)
# ----------------------------

# Ensure prices are not negative (a rare edge case)
predicted_prices[predicted_prices < 0] = 0

print("✅ Predictions generated and converted.")


# -----------------------------
# 7. Save Predictions to CSV
# -----------------------------
print(f"\nSaving predictions to {output_path}...")
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predicted_prices
})

results_df.to_csv(output_path, index=False)

print(f"\n🎉 Success! Predictions saved. You can find the file at '{output_path}'")
print("\nFirst 5 predictions:")
print(results_df.head())

Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted successfully!

Loading embeddings from /content/clip_multimodal_embeddings_test.pkl...
Loaded 100 embedding pairs.

Combining image and text embeddings...
Combined feature matrix shape: (100, 1024)

Loading LightGBM model from /content/drive/MyDrive/amazon/lgb_model.txt...
✅ Model loaded successfully!

Generating log predictions...
Converting predictions from log scale to actual prices...
✅ Predictions generated and converted.

Saving predictions to /content/price_predictions.csv...

🎉 Success! Predictions saved. You can find the file at '/content/price_predictions.csv'

First 5 predictions:
   sample_id  predicted_price
0     217392        50.460915
1     209156        20.253133
2     262333         5.175168
3     295979         8.899770
4      50604        10.004247


In [ ]:
# ======================================================
# CLIP Image & Text Embeddings for Test Dataset
# ======================================================

!pip install ftfy regex tqdm git+https://github.com/openai/CLIP.git > /dev/null

import torch
import clip
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
from io import BytesIO
import pickle

# -----------------------------
# 1. Load CLIP model with GPU verification
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# 🔍 Verify GPU availability
print("=" * 60)
print("🖥️  GPU CONFIGURATION")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Current GPU Memory Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
else:
    print("⚠️  WARNING: CUDA not available, using CPU (will be slower)")
print(f"Using device: {device}")
print("=" * 60)

model, preprocess = clip.load("ViT-B/32", device=device)
print(f"✅ CLIP model loaded on {device}\n")

# Verify model is actually on GPU
print(f"Model device: {next(model.parameters()).device}")
print("=" * 60 + "\n")

# -----------------------------
# 2. Load your test CSV
# -----------------------------
csv_path = "/content/drive/MyDrive/amazon/images_test/test.csv"
df = pd.read_csv(csv_path)

# --- Configuration ---
image_column = "image_link"
text_column = "catalog_content"
id_column = "sample_id"

print("Columns in CSV:", df.columns.tolist())
print(f"Using '{image_column}' for images and '{text_column}' for text.")
print(f"Total samples to process: {len(df)}\n")

# -----------------------------
# 3. Define helper to load image from URL
# -----------------------------
def load_image_from_url(url):
    """Fetches and opens an image from a URL."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image
    except Exception as e:
        print(f"Error loading image {url}: {e}")
        return None

# -----------------------------
# 4. Generate image and text embeddings
# -----------------------------
all_image_embeddings = []
all_text_embeddings = []
valid_ids = []
failed_count = 0

print("🚀 Starting embedding generation...")
for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    # --- Process Image ---
    img = load_image_from_url(row[image_column])
    if img is None:
        failed_count += 1
        continue

    image_input = preprocess(img).unsqueeze(0).to(device)

    # --- Process Text ---
    text_description = str(row[text_column])
    text_input = clip.tokenize([text_description], truncate=True).to(device)

    with torch.no_grad():
        # Encode both image and text
        img_features = model.encode_image(image_input)
        text_features = model.encode_text(text_input)

        # Normalize the features
        img_features /= img_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

    all_image_embeddings.append(img_features.cpu().numpy())
    all_text_embeddings.append(text_features.cpu().numpy())
    valid_ids.append(row.get(id_column, i))

    # Show GPU memory usage every 100 samples
    if device == "cuda" and (i + 1) % 100 == 0:
        print(f"\nGPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB allocated")

# -----------------------------
# 5. Save embeddings
# -----------------------------
image_embeddings_array = np.vstack(all_image_embeddings)
text_embeddings_array = np.vstack(all_text_embeddings)

data = {
    "image_embeddings": image_embeddings_array,
    "text_embeddings": text_embeddings_array,
    "sample_id": valid_ids
}

save_path = "/content/clip_multimodal_embeddings_test.pkl"
with open(save_path, "wb") as f:
    pickle.dump(data, f)

print("\n" + "=" * 60)
print("📊 SUMMARY")
print("=" * 60)
print(f"✅ Successfully processed: {len(valid_ids)} samples")
print(f"❌ Failed to load: {failed_count} samples")
print(f"💾 Saved to: {save_path}")
print(f"📏 Image embedding shape: {image_embeddings_array.shape}")
print(f"📏 Text embedding shape: {text_embeddings_array.shape}")
if device == "cuda":
    print(f"🎮 Final GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print("=" * 60)

  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-9i4t2wq7
🖥️  GPU CONFIGURATION
CUDA Available: True
CUDA Device: Tesla T4
CUDA Version: 12.6
Number of GPUs: 1
Current GPU Memory Allocated: 0.34 GB
Current GPU Memory Reserved: 0.70 GB
Using device: cuda
✅ CLIP model loaded on cuda

Model device: cuda:0

Columns in CSV: ['sample_id', 'catalog_content', 'image_link']
Using 'image_link' for images and 'catalog_content' for text.
Total samples to process: 75000

🚀 Starting embedding generation...


Processing:   0%|          | 101/75000 [00:09<2:53:38,  7.19it/s]


GPU Memory: 0.35 GB allocated


Processing:   0%|          | 201/75000 [00:19<1:55:33, 10.79it/s]


GPU Memory: 0.35 GB allocated


...


GPU Memory: 0.35 GB allocated


Processing:   4%|▍         | 3002/75000 [05:13<1:46:48, 11.24it/s]


GPU Memory: 0.35 GB allocated


Processing:   4%|▍         | 3070/75000 [05:21<2:05:36,  9.54it/s]


KeyboardInterrupt: 

In [ ]:
# ======================================================
# STEP 1: DOWNLOAD ALL IMAGES IN PARALLEL
# ======================================================

import pandas as pd
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

# ========================================
# CONFIGURATION - ADJUST THESE AS NEEDED
# ========================================
csv_path = "/content/drive/MyDrive/amazon/images_test/test.csv"
image_dir = Path("/content/downloaded_images")  # Local storage
image_column = "image_link"      # Column with image URLs
id_column = "sample_id"          # Column with unique IDs

MAX_WORKERS = 50                 # Number of parallel downloads
TIMEOUT = 5                      # Timeout per image (seconds)
# ========================================

# Create directory
image_dir.mkdir(exist_ok=True)

# Load CSV
print("📂 Loading CSV...")
df = pd.read_csv(csv_path)

print("\n" + "=" * 60)
print("📥 IMAGE DOWNLOAD CONFIGURATION")
print("=" * 60)
print(f"CSV: {csv_path}")
print(f"Total images: {len(df)}")
print(f"Save location: {image_dir}")
print(f"Image column: '{image_column}'")
print(f"ID column: '{id_column}'")
print(f"Parallel workers: {MAX_WORKERS}")
print(f"Timeout: {TIMEOUT}s")
print("=" * 60 + "\n")

# Verify columns exist
if image_column not in df.columns:
    print(f"❌ ERROR: Column '{image_column}' not found!")
    print(f"Available columns: {df.columns.tolist()}")
    raise ValueError(f"Column '{image_column}' not found in CSV")

if id_column not in df.columns:
    print(f"❌ ERROR: Column '{id_column}' not found!")
    print(f"Available columns: {df.columns.tolist()}")
    raise ValueError(f"Column '{id_column}' not found in CSV")

# Download function
def download_and_save(row):
    """Download image and save to disk"""
    url = row[image_column]
    img_id = row[id_column]
    save_path = image_dir / f"{img_id}.jpg"

    # Skip if already downloaded
    if save_path.exists():
        return img_id, True, "cached"

    try:
        response = requests.get(url, timeout=TIMEOUT)
        response.raise_for_status()

        with open(save_path, 'wb') as f:
            f.write(response.content)

        return img_id, True, None
    except Exception as e:
        return img_id, False, str(e)[:50]

# Download all images in parallel
print("🚀 Starting parallel download...\n")
start_time = time.time()
successful = []
failed = []
cached = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    futures = {executor.submit(download_and_save, row): idx
               for idx, row in df.iterrows()}

    # Process as they complete
    for future in tqdm(as_completed(futures), total=len(df), desc="Downloading"):
        img_id, success, error = future.result()

        if success:
            if error == "cached":
                cached += 1
            successful.append(img_id)
        else:
            failed.append((img_id, error))

download_time = time.time() - start_time

# Print summary
print("\n" + "=" * 60)
print("📥 DOWNLOAD SUMMARY")
print("=" * 60)
print(f"✅ Successfully downloaded: {len(successful) - cached}")
print(f"💾 Already cached: {cached}")
print(f"❌ Failed: {len(failed)}")
print(f"📊 Success rate: {len(successful)/len(df)*100:.1f}%")
print(f"⏱️  Total time: {download_time/60:.1f} minutes")
print(f"🚀 Download speed: {len(df)/download_time:.1f} images/second")
print(f"💾 Images saved to: {image_dir}")
print("=" * 60)

# Show sample failures
if failed:
    print(f"\n⚠️  Sample failures (first 10):")
    for img_id, error in failed[:10]:
        print(f"  - {img_id}: {error}")

# Save successful and failed IDs
successful_df = pd.DataFrame({'sample_id': successful})
successful_df.to_csv(image_dir / "successful_downloads.csv", index=False)
print(f"\n✅ Successful IDs saved to: {image_dir / 'successful_downloads.csv'}")

if failed:
    failed_df = pd.DataFrame(failed, columns=['sample_id', 'error'])
    failed_df.to_csv(image_dir / "failed_downloads.csv", index=False)
    print(f"❌ Failed IDs saved to: {image_dir / 'failed_downloads.csv'}")

# Disk usage check
total_size_mb = sum(f.stat().st_size for f in image_dir.glob("*.jpg")) / (1024 * 1024)
print(f"\n💾 Total disk space used: {total_size_mb:.1f} MB ({total_size_mb/1024:.2f} GB)")

print("\n" + "=" * 60)
print("✅ STEP 1 COMPLETE!")
print("=" * 60)
print(f"📁 {len(successful)} images ready for embedding")
print(f"📂 Location: {image_dir}")
print("\nNext: Run Step 2 to generate embeddings from these images")
print("=" * 60)

📂 Loading CSV...

📥 IMAGE DOWNLOAD CONFIGURATION
CSV: /content/drive/MyDrive/amazon/images_test/test.csv
Total images: 75000
Save location: /content/downloaded_images
Image column: 'image_link'
ID column: 'sample_id'
Parallel workers: 50
Timeout: 5s

🚀 Starting parallel download...



Downloading: 100%|██████████| 75000/75000 [08:03<00:00, 155.03it/s]



📥 DOWNLOAD SUMMARY
✅ Successfully downloaded: 74974
💾 Already cached: 0
❌ Failed: 26
📊 Success rate: 100.0%
⏱️  Total time: 8.5 minutes
🚀 Download speed: 147.1 images/second
💾 Images saved to: /content/downloaded_images

⚠️  Sample failures (first 10):
  - 123388: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 87830: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 164589: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 78086: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 258472: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 121811: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 93457: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 11395: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 35846: HTTPSConnectionPool(host='m.media-amazon.com', por
  - 15916: HTTPSConnectionPool(host='m.media-amazon.com', por

✅ Successful IDs saved to: /content/downloaded_images/successful_downloads.csv
❌ Failed IDs saved to: /content/downloaded

In [ ]:
# ======================================================
# STEP 1.5: RETRY FAILED IMAGE DOWNLOADS
# ======================================================

import pandas as pd
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

# ========================================
# CONFIGURATION - ADJUST THESE AS NEEDED
# ========================================
# Path to your ORIGINAL csv with all image links
original_csv_path = "/content/drive/MyDrive/amazon/images_test/test.csv"

# Path to the directory where images are saved and logs were created
image_dir = Path("/content/downloaded_images")

# Path to the CSV of failed downloads generated by the first script
failed_csv_path = image_dir / "failed_downloads.csv"

# Column names from your original CSV
image_column = "image_link"
id_column = "sample_id"

MAX_WORKERS = 20  # You can use a smaller number of workers for a retry
TIMEOUT = 10      # Consider increasing the timeout slightly for the retry
# ========================================

# --- SCRIPT LOGIC ---

# 1. Load original and failed dataframes
print("📂 Loading original data to find image links...")
original_df = pd.read_csv(original_csv_path)

print(f"📂 Loading failed downloads list from: {failed_csv_path}")
if not failed_csv_path.exists():
    print(f"❌ ERROR: Failed downloads file not found at '{failed_csv_path}'!")
    print("Please ensure the first script has run and generated this file.")
    exit()

failed_df = pd.read_csv(failed_csv_path)
failed_ids = failed_df[id_column].tolist()

# 2. Merge to get the URLs for the failed images
print(f"🔗 Finding links for the {len(failed_ids)} failed images...")
# Filter original_df to only include rows with failed IDs
retry_df = original_df[original_df[id_column].isin(failed_ids)].copy()

print("\n" + "=" * 60)
print("📥 RETRY DOWNLOAD CONFIGURATION")
print("=" * 60)
print(f"Images to retry: {len(retry_df)}")
print(f"Save location: {image_dir}")
print(f"Parallel workers: {MAX_WORKERS}")
print(f"Timeout: {TIMEOUT}s")
print("=" * 60 + "\n")

# 3. Define the download function (same as before)
def download_and_save(row):
    """Download image and save to disk"""
    url = row[image_column]
    img_id = row[id_column]
    save_path = image_dir / f"{img_id}.jpg"

    # We don't need to check for existing files, as these failed before
    try:
        response = requests.get(url, timeout=TIMEOUT)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

        with open(save_path, 'wb') as f:
            f.write(response.content)

        return img_id, True, None
    except Exception as e:
        return img_id, False, str(e)[:100] # Return a more detailed error

# 4. Run the download process for the retry list
print("🚀 Starting retry download...\n")
start_time = time.time()
successful_retries = []
still_failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_and_save, row): idx
               for idx, row in retry_df.iterrows()}

    for future in tqdm(as_completed(futures), total=len(retry_df), desc="Retrying"):
        img_id, success, error = future.result()

        if success:
            successful_retries.append(img_id)
        else:
            still_failed.append((img_id, error))

download_time = time.time() - start_time

# 5. Print the retry summary
print("\n" + "=" * 60)
print("📥 RETRY SUMMARY")
print("=" * 60)
print(f"✅ Successfully retried: {len(successful_retries)}")
print(f"❌ Still failed: {len(still_failed)}")
print(f"⏱️  Retry time: {download_time:.2f} seconds")
print("=" * 60)

if still_failed:
    print(f"\n⚠️  Images that FAILED AGAIN (first 10):")
    for img_id, error in still_failed[:10]:
        print(f"  - {img_id}: {error}")

    # Optionally, overwrite the failed_downloads.csv with the new list of failures
    still_failed_df = pd.DataFrame(still_failed, columns=[id_column, 'error'])
    still_failed_df.to_csv(image_dir / "still_failed_downloads.csv", index=False)
    print(f"\n❌ List of remaining failures saved to: {image_dir / 'still_failed_downloads.csv'}")

print("\n" + "=" * 60)
print("✅ RETRY SCRIPT COMPLETE!")
print("=" * 60)

📂 Loading original data to find image links...
📂 Loading failed downloads list from: /content/downloaded_images/failed_downloads.csv
🔗 Finding links for the 26 failed images...

📥 RETRY DOWNLOAD CONFIGURATION
Images to retry: 26
Save location: /content/downloaded_images
Parallel workers: 20
Timeout: 10s

🚀 Starting retry download...



Retrying: 100%|██████████| 26/26 [00:00<00:00, 1175.95it/s]


📥 RETRY SUMMARY
✅ Successfully retried: 25
❌ Still failed: 1
⏱️  Retry time: 0.42 seconds

⚠️  Images that FAILED AGAIN (first 10):
  - 286800: 404 Client Error: Not Found for url: https://m.media-amazon.com/images/I/813CjSgHj0S.jpg

❌ List of remaining failures saved to: /content/downloaded_images/still_failed_downloads.csv

✅ RETRY SCRIPT COMPLETE!


In [ ]:
!cp '/content/clip_combined_embeddings_test.pkl' '/content/drive/MyDrive/amazon'

In [ ]:
# ======================================================
# STEP 2: GENERATE AND COMBINE EMBEDDINGS
# ======================================================

import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm
import numpy as np
import warnings
import pickle

# Suppress annoying PIL decompression bomb warning
warnings.filterwarnings("ignore", category=UserWarning, message=".*Possibly corrupt EXIF data.*")
Image.MAX_IMAGE_PIXELS = None # Disable decompression bomb check

# ========================================
# CONFIGURATION - ADJUST THESE AS NEEDED
# ========================================
csv_path = "/content/drive/MyDrive/amazon/images_test/test.csv"
image_dir = Path("/content/downloaded_images")
output_path = Path("/content/clip_combined_embeddings_test.pkl") # Where to save the final embeddings

# Column names from your CSV
id_column = "sample_id"
text_column = "catalog_content"

# Model and batching settings
MODEL_NAME = "openai/clip-vit-base-patch32"
BATCH_SIZE = 128 # Adjust based on your GPU memory (e.g., 32, 64, 128)
# ========================================


# 1. SETUP MODEL AND DEVICE
# ------------------------------------------------------
print("🚀 Initializing setup...")
# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")

# Load CLIP model and processor
print(f"📦 Loading CLIP model: '{MODEL_NAME}'")
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME, use_fast=True)
print("✅ Model and processor loaded.")


# 2. LOAD AND PREPARE DATA
# ------------------------------------------------------
print(f"📂 Loading CSV from: {csv_path}")
df = pd.read_csv(csv_path)

# Handle potential missing text data
df[text_column] = df[text_column].fillna("").astype(str)

# Filter out rows where the corresponding image is missing
print("🖼️ Verifying downloaded images exist...")
initial_count = len(df)
image_paths = [image_dir / f"{img_id}.jpg" for img_id in df[id_column]]
df['image_path'] = image_paths
df = df[df['image_path'].apply(lambda p: p.exists())]
final_count = len(df)
print(f"Found {final_count} matching images out of {initial_count} CSV entries.")
if initial_count != final_count:
    print(f"⚠️  Skipped {initial_count - final_count} entries due to missing images.")


# 3. GENERATE EMBEDDINGS IN BATCHES
# ------------------------------------------------------
print("\n" + "=" * 60)
print("✨ Starting embedding generation...")
print("=" * 60)

all_sample_ids = []
all_image_embeddings = []
all_text_embeddings = []

# Process the dataframe in batches
for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Generating Embeddings"):
    batch_df = df.iloc[i:i+BATCH_SIZE]

    # Prepare batch data
    batch_texts = batch_df[text_column].tolist()
    batch_image_paths = batch_df['image_path'].tolist()
    batch_ids = batch_df[id_column].tolist()

    # Open images, handling potential errors
    batch_images = []
    valid_ids_in_batch = []
    for idx, path in enumerate(batch_image_paths):
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
            valid_ids_in_batch.append(batch_ids[idx])
        except Exception as e:
            print(f"❗️ Warning: Could not open image {path}. Skipping. Error: {e}")

    # Skip batch if no images were loaded successfully
    if not batch_images:
        continue

    # Preprocess images and text
    inputs = processor(
        text=batch_texts,
        images=batch_images,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    # Generate embeddings
    with torch.no_grad():
        outputs = model(**inputs)
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds

    # Normalize embeddings (good practice for similarity search)
    image_embeds /= image_embeds.norm(dim=-1, keepdim=True)
    text_embeds /= text_embeds.norm(dim=-1, keepdim=True)

    # Store results
    all_sample_ids.extend(valid_ids_in_batch)
    all_image_embeddings.append(image_embeds.cpu().numpy())
    all_text_embeddings.append(text_embeds.cpu().numpy())

# Concatenate all batch results into single numpy arrays
final_image_embeddings = np.vstack(all_image_embeddings)
final_text_embeddings = np.vstack(all_text_embeddings)


# 4. COMBINE AND SAVE RESULTS
# ------------------------------------------------------
# === MODIFIED SECTION TO COMBINE EMBEDDINGS ===
print("\n" + "=" * 60)
print("💾 COMBINING AND SAVING RESULTS AS .pkl")
print("=" * 60)

# 1. Combine image and text embeddings into a single vector for each sample
#    This stacks them side-by-side, e.g., (512,) + (512,) -> (1024,)
combined_embeddings = np.concatenate((final_image_embeddings, final_text_embeddings), axis=1)

# 2. Create the output dictionary with the combined embeddings
output_data = {
    'sample_ids': np.array(all_sample_ids),
    'combined_embeddings': combined_embeddings,
}

# 3. Save the dictionary to a .pkl file
with open(output_path, 'wb') as f:
    pickle.dump(output_data, f)
# === END OF MODIFIED SECTION ===


embedding_dim = combined_embeddings.shape[1]
print(f"✅ Successfully processed {len(all_sample_ids)} items.")
print(f"   - Combined embedding dimension: {embedding_dim}")
print(f"   - Combined embeddings shape: {combined_embeddings.shape}")
print(f"   - Sample IDs count: {len(all_sample_ids)}")
print(f"\n📦 Combined embeddings saved to: {output_path}")
print("\n" + "=" * 60)
print("✅ STEP 2 COMPLETE!")
print("=" * 60)

🚀 Initializing setup...
Using device: CUDA
📦 Loading CLIP model: 'openai/clip-vit-base-patch32'
✅ Model and processor loaded.
📂 Loading CSV from: /content/drive/MyDrive/amazon/images_test/test.csv
🖼️ Verifying downloaded images exist...
Found 74999 matching images out of 75000 CSV entries.
⚠️  Skipped 1 entries due to missing images.

✨ Starting embedding generation...



Generating Embeddings: 100%|██████████| 586/586 [1:14:16<00:00,  7.60s/it]



💾 COMBINING AND SAVING RESULTS AS .pkl
✅ Successfully processed 74999 items.
   - Combined embedding dimension: 1024
   - Combined embeddings shape: (74999, 1024)
   - Sample IDs count: 74999

📦 Combined embeddings saved to: /content/clip_combined_embeddings_test.pkl

✅ STEP 2 COMPLETE!


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
from google.colab import drive

# -----------------------------
# 1. Mount Google Drive
# -----------------------------
print("Mounting Google Drive...")
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive mounted successfully!")
except Exception as e:
    print(f"Error mounting drive: {e}")
    # Exit if drive mount fails
    exit()

# -----------------------------
# 2. Define File Paths
# -----------------------------
# This path should point to the file created by your generate_embeddings script
embeddings_path = "/content/clip_combined_embeddings_test.pkl"
model_path = "/content/drive/MyDrive/amazon/lgb_model.txt"
output_path = "/content/drive/MyDrive/amazon/price_predictions.csv"


# Wrap the main logic in a try-except block for better error handling
try:
    # -----------------------------
    # 3. Load Embeddings
    # -----------------------------
    print(f"\nLoading embeddings from {embeddings_path}...")
    with open(embeddings_path, 'rb') as f:
        data = pickle.load(f)

    # --- FIX ---
    # The script now loads the pre-combined embeddings directly from the pickle file.
    # It also uses the correct 'sample_ids' key as shown in your error message.
    print("Available keys in the pickle file:", data.keys())
    X_test = data["combined_embeddings"]
    sample_ids = data["sample_ids"]
    # ----------------------------------------

    print(f"Loaded {len(sample_ids)} combined embeddings.")


    # -----------------------------
    # 4. (No longer needed) Combine Embeddings into a Single Feature Set
    # -----------------------------
    # The embeddings are already combined, so we can skip this step.
    print("\nEmbeddings are already combined.")
    print("Feature matrix shape:", X_test.shape)


    # -----------------------------
    # 5. Load the Pre-trained LightGBM Model
    # -----------------------------
    print(f"\nLoading LightGBM model from {model_path}...")
    bst = lgb.Booster(model_file=model_path)
    print("✅ Model loaded successfully!")


    # -----------------------------
    # 6. Make Predictions & Convert from Log Scale
    # -----------------------------
    print("\nGenerating log predictions...")
    log_predictions = bst.predict(X_test)

    # Convert log predictions back to the original price scale
    print("Converting predictions from log scale to actual prices...")
    predicted_prices = np.expm1(log_predictions)

    # Ensure prices are not negative (a rare edge case)
    predicted_prices[predicted_prices < 0] = 0
    print("✅ Predictions generated and converted.")


    # -----------------------------
    # 7. Save Predictions to CSV
    # -----------------------------
    print(f"\nSaving predictions to {output_path}...")
    results_df = pd.DataFrame({
        'sample_id': sample_ids,
        'price': predicted_prices
    })

    results_df.to_csv(output_path, index=False)

    print(f"\n🎉 Success! Predictions saved. You can find the file at '{output_path}'")
    print("\nFirst 5 predictions:")
    print(results_df.head())

except FileNotFoundError as e:
    print(f"\n--- File Not Found Error ---")
    print(f"Could not find a file: {e.filename}")
    print("Please double-check your 'embeddings_path' and 'model_path' variables.")

except KeyError as e:
    print(f"\n--- Key Error ---")
    print(f"The key {e} was not found in your pickle file.")
    print("This script now expects 'combined_embeddings' and 'sample_ids'.")
    print("Please check the 'Available keys' printout to confirm the names are correct.")

except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")



Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted successfully!

Loading embeddings from /content/clip_combined_embeddings_test.pkl...
Available keys in the pickle file: dict_keys(['sample_ids', 'combined_embeddings'])
Loaded 74999 combined embeddings.

Embeddings are already combined.
Feature matrix shape: (74999, 1024)

Loading LightGBM model from /content/drive/MyDrive/amazon/lgb_model.txt...
✅ Model loaded successfully!

Generating log predictions...
Converting predictions from log scale to actual prices...
✅ Predictions generated and converted.

Saving predictions to /content/drive/MyDrive/amazon/price_predictions.csv...

🎉 Success! Predictions saved. You can find the file at '/content/drive/MyDrive/amazon/price_predictions.csv'

First 5 predictions:
   sample_id      price
0     100179  15.207007
1     245611  22.863407
2     146263  20.796078
3      95658   9.305906
4      36806  21.385558
